In [3]:
from math import e

import numpy as np
import pandas as pd
import os
import json
from tqdm import tqdm

def sequential_split(interaction_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_parts, test_parts = [], []
    for u, g in interaction_df.groupby("user_id"):
        g = g.sort_values(["timestamp", "item_id"], kind="mergesort")
        if len(g) <= 1:
            continue
        test_parts.append(g.iloc[[-1]])
        train_parts.append(g.iloc[:-1])

    train_df = pd.concat(train_parts, ignore_index=True)
    test_df  = pd.concat(test_parts,  ignore_index=True)
    return train_df, test_df


raw_data_path = '../SHNS_data_raw'
output_data_path = '../SHNS_data'
dataset_path = raw_data_path + '/' + 'Gowalla_totalCheckins.txt'
df = pd.read_csv(dataset_path, sep="\t", header=None, names=["user_id", "timestamp", "latitude", "longitude", "item_id"])
ts = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
df = df[ts.notna()].copy()
df["timestamp"] = (ts.view("int64") // 10**9).astype("int64")

df = df.drop_duplicates(subset=["user_id", "item_id"])

while True:
    prev_len = len(df)
    df = df[df.groupby("user_id")["item_id"].transform("size") >= 5].copy()
    df = df[df.groupby("item_id")["user_id"].transform("size") >= 5].copy()
    if len(df) == prev_len:
        break

users = np.sort(df["user_id"].dropna().unique())
items = np.sort(df["item_id"].dropna().unique())
user2id = {u:i for i, u in enumerate(users)}
item2id = {a:i for i, a in enumerate(items)}

user_ids = df["user_id"].map(user2id).astype("int64")
item_ids = df["item_id"].map(item2id).astype("int64")
timestamps = df["timestamp"].astype("int64")
interaction_df = pd.DataFrame({"user_id": user_ids, "item_id": item_ids, "timestamp": timestamps})
train_valid_df, test_df = sequential_split(interaction_df)
train_df, valid_df = sequential_split(train_valid_df)

cur_output_data_path = output_data_path + '/' + 'gowalla_seq'
os.makedirs(cur_output_data_path, exist_ok=True)
if os.path.exists(cur_output_data_path + "/train.txt"):
    print(f"{cur_output_data_path + '/train.txt'} already exists. Checking sanity...")
    existing_train_df = pd.read_csv(cur_output_data_path + "/train.txt", sep="\t", header=None, names=["user_id", "item_id"])
    assert train_df[["user_id", "item_id"]].equals(existing_train_df), "Existing train.txt does not match the newly created train_df!"
else:
    train_df[["user_id", "item_id"]].to_csv(cur_output_data_path + "/train.txt", sep="\t", header=False, index=False)
if os.path.exists(cur_output_data_path + "/valid.txt"):
    print(f"{cur_output_data_path + '/valid.txt'} already exists. Checking sanity...")
    existing_valid_df = pd.read_csv(cur_output_data_path + "/valid.txt", sep="\t", header=None, names=["user_id", "item_id"])
    assert valid_df[["user_id", "item_id"]].equals(existing_valid_df), "Existing valid.txt does not match the newly created valid_df!"
else:
    valid_df[["user_id", "item_id"]].to_csv(cur_output_data_path + "/valid.txt", sep="\t", header=False, index=False)
if os.path.exists(cur_output_data_path + "/test.txt"):
    print(f"{cur_output_data_path + '/test.txt'} already exists. Checking sanity...")
    existing_test_df = pd.read_csv(cur_output_data_path + "/test.txt", sep="\t", header=None, names=["user_id", "item_id"])
    assert test_df[["user_id", "item_id"]].equals(existing_test_df), "Existing test.txt does not match the newly created test_df!"
else:
    test_df[["user_id", "item_id"]].to_csv(cur_output_data_path + "/test.txt",  sep="\t", header=False, index=False)


C:\Users\Junha\AppData\Local\Temp\ipykernel_4676\2417818183.py:29: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  df["timestamp"] = (ts.view("int64") // 10**9).astype("int64")
